## Pydantic Output Parser

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_core.output_parsers import PydanticOutputParser

In [4]:
from langchain_core.prompts import PromptTemplate

In [6]:
from pydantic import BaseModel,Field

In [7]:
class Person(BaseModel):
    name : str = Field(description='the name of the person')
    age : int = Field(gt = 18 , description='the age of the person')
    country : str = Field(description= ' the country name of the person it belogs to ')

In [8]:
parser = PydanticOutputParser(pydantic_object=Person)

In [ ]:
prompt = PromptTemplate(
    template = 'give me the information about the the this batsman {batsman}\n {format_instructions}',
    input_variables =['batsman'],
    partial_variables = {'format_instructions':parser.get_format_instructions}
)

In [11]:
prompt =prompt.invoke(input ={'batsman':'joe root'})

In [12]:

print(prompt)

text='give me the information about the the this batsman joe root\n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "the name of the person", "title": "Name", "type": "string"}, "age": {"description": "the age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "country": {"description": " the country name of the person it belogs to ", "title": "Country", "type": "string"}}, "required": ["name", "age", "country"]}\n```'


In [13]:
model = ChatOpenAI()

### 1 way to do things manually with out chains

In [14]:
response = model.invoke(prompt).content

In [15]:
print(response)

{
  "name": "Joe Root",
  "age": 31,
  "country": "England"
}


In [16]:
final_output = parser.invoke(response)

In [17]:
print(final_output)

name='Joe Root' age=31 country='England'


## 2nd way with the help of the chains

In [19]:
prompt = PromptTemplate(
    template = 'give me the information about the the this batsman {batsman}\n {format_instructions}',
    input_variables =['batsman'],
    partial_variables = {'format_instructions':parser.get_format_instructions}
)

In [20]:
chain = prompt | model | parser

In [21]:
chain.invoke(input={'batsman':'steve smith'})

Person(name='Steve Smith', age=31, country='Australia')

## using these parsers we can really validate the output also